# Sākṣī — Training the on-device crack detector

This notebook produces the model file the app is already wired to receive:
**`crack-seg.pte`** (ExecuTorch), plus an ONNX fallback. It also prints the real
**`mAP50`** you must paste into `services/ai/yoloEngine.ts` — the honest accuracy
number the UI shows. Nothing here rounds it up.

**Runtime:** Colab free **T4 GPU** is enough. Set it first:
`Runtime → Change runtime type → Hardware accelerator → T4 GPU`.

**Time:** ~1.5–3 h for 80 epochs. Start it and let it run.

**You do not need a dataset.** Ultralytics auto-downloads the public 4,029-image
`crack-seg` instance-segmentation set. See `docs/DAMAGE-MODEL.md` for how this
slots into the app.

## Step 0 — Confirm the GPU is on

If this prints `NVIDIA-SMI ... Tesla T4`, you're set. If it errors, you're on CPU
— fix the runtime type above before training, or 80 epochs will take all day.

In [ ]:
!nvidia-smi

## Step 1 — Install Ultralytics

This pulls in YOLOv8 only. **Do not install `executorch` here** — its wheel drags
in a torch build that mismatches Colab's pre-compiled torchvision, which breaks
torchvision's `nms` op and kills training with
`RuntimeError: operator torchvision::nms does not exist`. The ExecuTorch export
is handled separately in Step 5, *after* your model is trained and safe.

The check below must print a function for `torchvision.ops.nms`. If it errors,
**Runtime → Restart session** and run this cell again before anything else.

In [ ]:
!pip install -q -U ultralytics

import ultralytics, torch, torchvision
ultralytics.checks()
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| cuda', torch.cuda.is_available())
# Guard: this is the exact op that fails on a torch/torchvision mismatch.
print('nms op OK →', torchvision.ops.nms)

## Step 2 — Choose the task

The app's overlay draws **boxes**, so a plain **detection** model is the simplest
faithful match and exports most reliably. Segmentation gives masks the current UI
would discard — but it's what `docs/DAMAGE-MODEL.md` documents, and it still
yields boxes.

Pick one below. Detection (`detect`) is the recommended default; flip to
`segment` only if you want masks for a later UI. Everything downstream adapts.

In [ ]:
# 'detect' → boxes only, simplest export, matches the current overlay (recommended)
# 'segment' → boxes + masks (what DAMAGE-MODEL.md documents)
TASK = 'detect'

EPOCHS = 80
IMGSZ = 640
DATA = 'crack-seg.yaml'   # public 4,029-image crack set, auto-downloaded

BASE_MODEL = 'yolov8n.pt' if TASK == 'detect' else 'yolov8n-seg.pt'
RUN_NAME = f'crack-{TASK}'
print(f'Training {BASE_MODEL} ({TASK}) on {DATA} for {EPOCHS} epochs @ {IMGSZ}px')

## Step 3 — Train

Watch **`mAP50`** in the per-epoch table — that is the number that ends up in the
app. `patience=20` stops early if it plateaus, so you rarely wait the full 80
epochs. Weights land in `runs/<task>/<RUN_NAME>/weights/best.pt`.

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)
results = model.train(
    task=TASK,
    data=DATA,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=16,
    patience=20,
    name=RUN_NAME,
    plots=True,
)

## Step 4 — Read the honest mAP50

This validates `best.pt` on the held-out split and prints the exact `mAP50` to
paste into `yoloEngine.ts`. Use the **box** mAP50 — the app reports boxes.
**Copy this number verbatim. Do not round it up.**

In [ ]:
from pathlib import Path

best = Path('runs') / TASK / RUN_NAME / 'weights' / 'best.pt'
assert best.exists(), f'No weights at {best} — did training finish?'

metrics = YOLO(str(best)).val(data=DATA, imgsz=IMGSZ)
map50 = float(metrics.box.map50)   # box mAP@0.50 — this is what the UI shows
map5095 = float(metrics.box.map)   # box mAP@0.50:0.95, for your own reference

print('=' * 52)
print(f'  mAP50   = {map50:.4f}   ← paste this into DAMAGE_MODEL.info.mAP50')
print(f'  mAP50-95= {map5095:.4f}   (reference only)')
print('=' * 52)

## Step 5 — Export the model

**ONNX first, always.** It's rock-solid, needs only the healthy torch you trained
with, and is ~90% of the value — pair it with `onnxruntime-react-native` in the
app. This cell produces `crack-seg.onnx` and never touches your session.

The ExecuTorch `.pte` is a **separate, optional** cell below, because installing
the `executorch` wheel re-triggers the same torch/torchvision clash that broke
training. So we get the ONNX file safely in hand *first*; only then do you risk
the `.pte`. If it corrupts the session, you've lost nothing — download the ONNX
zip and ship that.

In [ ]:
import shutil
from pathlib import Path

out = Path('/content/saksi-model')
out.mkdir(exist_ok=True)
m = YOLO(str(best))

# Safe, reliable export — torch is still healthy here.
path = m.export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True)
dest = out / 'crack-seg.onnx'
shutil.copy(path, dest)
shutil.copy(best, out / 'best.pt')   # keep source weights to re-export later
runtime, exported = 'onnx', dest
print(f'✓ ONNX export → {dest} ({dest.stat().st_size/1e6:.1f} MB)')
print(f'  runtime = {runtime}  (use onnxruntime-react-native in the app)')

## Step 5b — ExecuTorch `.pte` (optional, run last)

Only bother with this if you specifically want the `executorch` runtime instead of
onnxruntime. **Run Step 7 and download the ONNX zip first** — installing the
`executorch` wheel can break torch in this session (the same `nms` error), and if
it does you'll want the ONNX already on your machine.

If it succeeds you'll have `crack-seg.pte`; if it fails, ignore it and ship the
ONNX. Either path gives the app a real, working model.

## Step 6 — Sanity-check inference on a real image

Runs `best.pt` on a couple of validation images and shows the boxes it finds.
This is your proof the model actually detects cracks before you ship it — if this
is blank, don't drop it in and claim it works.

In [ ]:
import glob, os
from IPython.display import Image as IPyImage, display

# crack-seg unpacks under the Ultralytics datasets dir; find a few val images.
candidates = sorted(glob.glob('/root/datasets/crack-seg/**/val/**/*.jpg', recursive=True))
candidates += sorted(glob.glob('/content/datasets/crack-seg/**/val/**/*.jpg', recursive=True))
sample = candidates[:2]
print('Testing on:', sample or 'no val images found — check dataset path')

if sample:
    preds = YOLO(str(best)).predict(sample, imgsz=IMGSZ, conf=0.35, save=True)
    save_dir = preds[0].save_dir
    for p in sorted(glob.glob(os.path.join(save_dir, '*.jpg')))[:2]:
        display(IPyImage(filename=p, width=480))

## Step 7 — Download the model

Zips the exported model + weights + a `README` recording the real `mAP50`, then
triggers a browser download. Unzip and follow Step 8.

In [ ]:
readme = out / 'MODEL-CARD.txt'
readme.write_text(
    f'Sakshi crack detector\n'
    f'task     : {TASK}\n'
    f'base     : {BASE_MODEL}\n'
    f'dataset  : {DATA} (public crack set)\n'
    f'epochs   : {EPOCHS} @ {IMGSZ}px\n'
    f'runtime  : {runtime}\n'
    f'mAP50    : {map50:.4f}   <-- paste into DAMAGE_MODEL.info.mAP50\n'
    f'mAP50-95 : {map5095:.4f}\n'
    f'file     : {exported.name}\n'
)

zip_path = shutil.make_archive('/content/saksi-model', 'zip', str(out))
print('Zipped →', zip_path)

from google.colab import files
files.download(zip_path)

## Step 8 — Drop it into the app

Full recipe in `docs/DAMAGE-MODEL.md`. In short:

1. **If you got `crack-seg.pte` (ExecuTorch):** put it at
   `assets/models/crack-seg.pte`. Keep runtime `'executorch'`.
   **If you got `crack-seg.onnx` (fallback):** put it at
   `assets/models/crack-seg.onnx`, and switch `services/ai/executorch.ts` to
   `onnxruntime-react-native`, and set `runtime: 'onnx'` below.

2. In `services/ai/yoloEngine.ts`, flip the config seam (paste your real mAP50):

```ts
export const DAMAGE_MODEL = {
  source: require('../../assets/models/crack-seg.pte'), // was null
  info: {
    name: 'YOLOv8n crack detector',
    version: '0.1.0',
    classes: ['crack'],
    mAP50: 0.____,          // ← the mAP50 printed in Step 4
    runtime: 'executorch',  // 'onnx' if you used the fallback
  },
};
```

3. Install the runtime and rebuild the dev client (native module — not Expo Go):

```bash
npm i react-native-executorch
npx expo prebuild
eas build --profile development --platform android   # or ios
```

4. On device: open an observation → **Scan photo for damage**. If boxes look
   shifted/scaled, adjust the one marked `imageSize()` / `normalizeBbox()` line in
   `useExecutorchDamageDetector` — the single untested coordinate-space assumption.

**Honesty rules that must not regress** (from `docs/DAMAGE-MODEL.md`): candidates,
not verdicts; the model fills *what*, the human sets *how urgent*; no invented
numbers; provenance recorded as `ai_assisted = 1`; the manual report path always
works without the model.

In [ ]:
# Optional: attempt the ExecuTorch .pte export. May corrupt the session's torch.
!pip install -q executorch 2>/dev/null || echo 'executorch wheel unavailable'

try:
    m_pte = YOLO(str(best))
    path = m_pte.export(format='executorch', imgsz=IMGSZ)
    src = Path(path)
    pte = next(src.rglob('*.pte')) if src.is_dir() else (
        src if src.suffix == '.pte' else next(src.parent.rglob('*.pte')))
    dest_pte = out / 'crack-seg.pte'
    shutil.copy(pte, dest_pte)
    runtime, exported = 'executorch', dest_pte
    print(f'✓ ExecuTorch export → {dest_pte} ({dest_pte.stat().st_size/1e6:.1f} MB)')
    print('  Re-run Step 7 to download the zip (now including the .pte).')
except Exception as e:
    print(f'✗ ExecuTorch export failed: {e}')
    print('→ No problem. Ship the ONNX from Step 5 with onnxruntime-react-native.')